<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: Preprocesamiento y Análisis Espectral Avanzado
**Proyecto:** Detección de Deslizamientos mediante IA | **Universidad EAFIT** 
**Objetivo:** Configurar la pipeline de datos, aplicar Data Augmentation y realizar un análisis de firmas espectrales para validar la separabilidad de las clases.

In [ ]:
import os, sys, h5py, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ── CONFIGURACIÓN DE RUTAS ──────────────────────────────────────────────────
DATA_ROOT = Path('/content/landslide4sense')
img_dir = DATA_ROOT / "TrainData" / "img"
mask_dir = DATA_ROOT / "TrainData" / "mask"
img_list = sorted(list(img_dir.glob("*.h5")))
mask_list = sorted(list(mask_dir.glob("*.h5")))

CHANNEL_NAMES = ['SAR VV', 'SAR VH', 'SAR VV/VH', 'B2-Blue', 'B3-Green', 'B4-Red', 
                 'B5-RE1', 'B6-RE2', 'B7-RE3', 'B8-NIR', 'B8A-RE4', 'B11-SWIR1', 'B12-SWIR2', 'DEM']

print(f"✓ Dataset cargado: {len(img_list)} muestras.")

## 1. Pipeline de Datos con Fix de Memoria
Definimos la clase `LandslideDataset` con normalización Z-Score y aumentación geométrica, asegurando que los arreglos sean contiguos para PyTorch.

In [ ]:
def zscore_norm(patch):
    mean = np.mean(patch, axis=(0, 1))
    std = np.std(patch, axis=(0, 1)) + 1e-6
    return (patch - mean) / std

class LandslideDataset(Dataset):
    def __init__(self, img_files, mask_files, augment=False):
        self.img_files = img_files
        self.mask_files = mask_files
        self.augment = augment

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        with h5py.File(self.img_files[idx], 'r') as hf: 
            img = hf[list(hf.keys())[0]][()].astype(np.float32)
        with h5py.File(self.mask_files[idx], 'r') as hf: 
            mask = hf[list(hf.keys())[0]][()].astype(np.float32)
        
        img = zscore_norm(img)
        
        if self.augment:
            k = np.random.randint(0, 4)
            img = np.rot90(img, k)
            mask = np.rot90(mask, k)
            if np.random.random() > 0.5:
                img = np.flip(img, axis=1)
                mask = np.flip(mask, axis=1)

        # Transpose y Fix de Memoria Contigua
        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        mask = np.ascontiguousarray(mask)

        return torch.from_numpy(img).float(), torch.from_numpy(mask).long()

## 2. Análisis Espectral y de Correlación
Evaluamos la firma de los deslizamientos y la redundancia entre los 14 canales de entrada.

In [ ]:
def plot_advanced_analysis(ds, idx):
    img_t, mask_t = ds[idx]
    img, mask = img_t.numpy(), mask_t.numpy()
    
    # --- Matriz de Correlación ---
    flat_data = img.reshape(14, -1).T
    df = pd.DataFrame(flat_data, columns=CHANNEL_NAMES)
    corr = df.corr()

    # --- Perfil Espectral ---
    feat_landslide = img[:, mask == 1].mean(axis=1) if (mask == 1).any() else np.zeros(14)
    feat_stable = img[:, mask == 0].mean(axis=1) if (mask == 0).any() else np.zeros(14)

    # --- Dashboard Visual ---
    fig = plt.figure(figsize=(22, 12))
    plt.style.use('seaborn-v0_8-whitegrid')

    # 1. Correlación
    ax1 = plt.subplot2grid((2, 2), (0, 0))
    sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, ax=ax1)
    ax1.set_title("Correlación entre Bandas", fontweight='bold')

    # 2. Firma Espectral
    ax2 = plt.subplot2grid((2, 2), (0, 1))
    ax2.plot(CHANNEL_NAMES, feat_landslide, 'r-o', label='Deslizamiento', markersize=8)
    ax2.plot(CHANNEL_NAMES, feat_stable, 'g-s', label='Estable', markersize=8, alpha=0.5)
    ax2.set_xticklabels(CHANNEL_NAMES, rotation=45)
    ax2.set_title("Firma Espectral Promedio (Z-Score)", fontweight='bold')
    ax2.legend()

    # 3. Contexto RGB
    ax3 = plt.subplot2grid((2, 2), (1, 0))
    rgb = img[[5, 4, 3]].transpose(1, 2, 0)
    p2, p98 = np.percentile(rgb, (2, 98))
    ax3.imshow(np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1))
    ax3.set_title("Muestra RGB (Bands 4,3,2)", fontweight='bold')
    ax3.axis('off')

    # 4. Máscara
    ax4 = plt.subplot2grid((2, 2), (1, 1))
    ax4.imshow(mask, cmap='jet')
    ax4.set_title("Máscara (GT)", fontweight='bold')
    ax4.axis('off')

    plt.tight_layout()
    plt.show()

## 3. Validación Final
Buscamos un parche representativo y ejecutamos el análisis.

In [ ]:
ds = LandslideDataset(img_list, mask_list, augment=True)

# Buscador de parches con datos reales
idx_interesting = 0
for i in range(1000, 2000):
    _, m = ds[i]
    if m.sum() > 500: # Al menos 500 píxeles de deslizamiento
        idx_interesting = i
        break

plot_advanced_analysis(ds, idx_interesting)